In [13]:
%pip install "weaviate-client>=4.24,<5" "numpy>=1.26,<3" "pymilvus>=2.6,<3" "pinecone>=7,<8"

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip
C:\Users\hienm\AppData\Roaming\Python\Python313\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedWriter name=3>
  res = process_handler(cmd, _system_body)
C:\Users\hienm\AppData\Roaming\Python\Python313\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=4>
  res = process_handler(cmd, _system_body)
C:\Users\hienm\AppData\Roaming\Python\Python313\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=5>
  res = process_handler(cmd, _system_body)


# Weaviate Vector Database: từ dữ liệu đến tìm kiếm vector

## Mục tiêu seminar
- Hiểu collection, schema/property, object UUID và vector được lưu như thế nào trong Weaviate.
- Thấy text2vec-transformers tạo vector từ title/content và HNSW phục vụ tìm kiếm láng giềng gần.
- Thực hành batch insert, CRUD, thống kê, đo Recall@10/latency của HNSW và hybrid search.
- Thực hành retrieval cho RAG không cần LLM, đồng thời benchmark cùng vector trên Weaviate, Milvus và Pinecone (Pinecone tùy chọn).

## Khởi động Weaviate
Từ thư mục chứa docker-compose.yml:

```powershell
docker compose up -d
docker compose ps
```

Đợi cả Weaviate và t2v-transformers hoạt động rồi chạy notebook từ trên xuống. Jupyter trên máy host dùng mặc định 127.0.0.1:8080 và 127.0.0.1:50051. Model embedding all-MiniLM-L6-v2 phù hợp hơn với tiếng Anh; đây là giới hạn cần nêu khi đánh giá dữ liệu tiếng Việt.

## RAG không cần LLM
Notebook chỉ minh họa phần retrieval: Weaviate tìm tài liệu liên quan, áp dụng metadata filter và dựng context kèm nguồn. Bước sinh câu trả lời bằng LLM là tùy chọn của một ứng dụng RAG hoàn chỉnh, không cần cho demo vector database này.

## Bật Milvus local để so sánh (tùy chọn)

```powershell
docker compose --profile comparison up -d
```

Milvus standalone cần thêm etcd, MinIO và dung lượng Docker. Notebook so sánh cùng một bộ vector tổng hợp trên Weaviate và Milvus; bộ SDK được cài ở cell đầu. Pinecone là cloud, dùng key qua biến môi trường và chỉ chạy khi bật cờ rõ ràng; có thể phát sinh chi phí.

**Kịch bản 15–20 phút:** schema/HNSW → dữ liệu và vector → CRUD/aggregate → benchmark recall-latency → truy vấn và đánh giá hybrid → retrieval/context cho RAG → so sánh database. Benchmark nhỏ trong notebook nhằm minh họa, không đại diện cho tải production.

## Cấu hình môi trường tùy chọn cho các benchmark

In [14]:
import os

# Để bật benchmark Pinecone, đặt PINECONE_API_KEY trước khi mở Jupyter,
# sau đó đặt RUN_PINECONE_COMPARISON=1. Không lưu key trong notebook.
RUN_PINECONE_COMPARISON = os.getenv("RUN_PINECONE_COMPARISON", "0") == "1"
PINECONE_CLOUD = os.getenv("PINECONE_CLOUD", "aws")
PINECONE_REGION = os.getenv("PINECONE_REGION", "us-east-1")

print(f"Milvus local: 127.0.0.1:19530 (bật profile comparison để chạy)")
print(f"Pinecone cloud benchmark: {'enabled' if RUN_PINECONE_COMPARISON else 'disabled'}")

Milvus local: 127.0.0.1:19530 (bật profile comparison để chạy)
Pinecone cloud benchmark: disabled


## 1. Kết nối và kiểm tra Weaviate

In [15]:
import os
import weaviate

WEAVIATE_HOST = os.getenv("WEAVIATE_HOST", "127.0.0.1")
WEAVIATE_PORT = int(os.getenv("WEAVIATE_PORT", "8080"))
WEAVIATE_GRPC_PORT = int(os.getenv("WEAVIATE_GRPC_PORT", "50051"))
COLLECTION_NAME = "SeminarTechArticle"

def init_weaviate_client():
    client = weaviate.connect_to_local(
        host=WEAVIATE_HOST,
        port=WEAVIATE_PORT,
        grpc_port=WEAVIATE_GRPC_PORT,
    )
    if not client.is_ready():
        client.close()
        raise RuntimeError(
            f"Weaviate chưa sẵn sàng tại {WEAVIATE_HOST}:{WEAVIATE_PORT}. "
            "Chạy docker compose up -d và kiểm tra docker compose ps."
        )

    server_meta = client.get_meta()
    modules = ", ".join(server_meta.get("modules", {}).keys())
    print(f"Weaviate {server_meta['version']} sẵn sàng tại {WEAVIATE_HOST}:{WEAVIATE_PORT}")
    print(f"Module đang bật: {modules}")
    return client

client = init_weaviate_client()

Weaviate 1.27.0 sẵn sàng tại 127.0.0.1:8080
Module đang bật: text2vec-transformers


## 2. Collection, schema và HNSW index

In [16]:
import weaviate.classes.config as wvc

def setup_schema(client: weaviate.WeaviateClient):
    # Dùng tên riêng của seminar, không xóa hoặc sửa collection khác trong server.
    if client.collections.exists(COLLECTION_NAME):
        print(f"Dùng collection seminar hiện có: {COLLECTION_NAME}")
        return client.collections.get(COLLECTION_NAME)

    collection = client.collections.create(
        name=COLLECTION_NAME,
        vector_config=wvc.Configure.Vectors.text2vec_transformers(
            source_properties=["title", "content"],
            vector_index_config=wvc.Configure.VectorIndex.hnsw(
                distance_metric=wvc.VectorDistances.COSINE,
                max_connections=32,
                ef_construction=128,
                ef=64,
            ),
        ),
        properties=[
            wvc.Property(name="title", data_type=wvc.DataType.TEXT),
            wvc.Property(name="content", data_type=wvc.DataType.TEXT),
            wvc.Property(name="category", data_type=wvc.DataType.TEXT),
            wvc.Property(name="published_year", data_type=wvc.DataType.INT),
        ],
    )
    print(f"Đã tạo collection {COLLECTION_NAME} với vectorizer text2vec-transformers")
    return collection

tech_collection = setup_schema(client)

Dùng collection seminar hiện có: SeminarTechArticle


## 3. Kiểm tra schema và cấu hình vector index

## 4. Benchmark HNSW: cấu hình, recall@10 và latency
So sánh HNSW với M khác nhau và Flat index làm baseline exact trên cùng vector tổng hợp. Recall@10 là tỷ lệ láng giềng exact tìm thấy; median/p95 là latency truy vấn. Đây là microbenchmark nhỏ, phụ thuộc máy và cache, không dùng để suy rộng thành kết quả production.

In [17]:
import statistics
import time
import uuid

import numpy as np

BENCH_DIMENSION = 384
BENCH_SIZE = 1500
BENCH_QUERY_COUNT = 24
BENCH_TOP_K = 10
BENCH_NAMESPACE = uuid.UUID("333ce044-09e6-4f98-b236-086cecc90133")

rng = np.random.default_rng(42)
BENCH_VECTORS = rng.standard_normal((BENCH_SIZE, BENCH_DIMENSION), dtype=np.float32)
BENCH_VECTORS /= np.linalg.norm(BENCH_VECTORS, axis=1, keepdims=True)
BENCH_QUERY_VECTORS = rng.standard_normal(
    (BENCH_QUERY_COUNT, BENCH_DIMENSION), dtype=np.float32
)
BENCH_QUERY_VECTORS /= np.linalg.norm(BENCH_QUERY_VECTORS, axis=1, keepdims=True)
exact_scores = BENCH_QUERY_VECTORS @ BENCH_VECTORS.T
exact_top_k = np.argsort(-exact_scores, axis=1)[:, :BENCH_TOP_K]
BENCH_IDS = [str(uuid.uuid5(BENCH_NAMESPACE, str(index))) for index in range(BENCH_SIZE)]

index_variants = [
    ("HNSW M=8", wvc.Configure.VectorIndex.hnsw(
        distance_metric=wvc.VectorDistances.COSINE, max_connections=8,
        ef_construction=128, ef=64,
    )),
    ("HNSW M=16", wvc.Configure.VectorIndex.hnsw(
        distance_metric=wvc.VectorDistances.COSINE, max_connections=16,
        ef_construction=128, ef=64,
    )),
    ("HNSW M=32", wvc.Configure.VectorIndex.hnsw(
        distance_metric=wvc.VectorDistances.COSINE, max_connections=32,
        ef_construction=128, ef=64,
    )),
    ("Flat exact", wvc.Configure.VectorIndex.flat(
        distance_metric=wvc.VectorDistances.COSINE,
    )),
]

benchmark_results = []
created_benchmark_collections = []
try:
    for variant, index_config in index_variants:
        collection_name = f"SeminarHnsw{uuid.uuid4().hex[:10]}"
        benchmark_collection = client.collections.create(
            name=collection_name,
            vector_config=wvc.Configure.Vectors.self_provided(
                name="bench",
                vector_index_config=index_config,
            ),
            properties=[wvc.Property(name="label", data_type=wvc.DataType.TEXT)],
        )
        created_benchmark_collections.append(collection_name)

        build_started = time.perf_counter()
        with benchmark_collection.batch.dynamic() as batch:
            for index, vector in enumerate(BENCH_VECTORS):
                batch.add_object(
                    uuid=BENCH_IDS[index],
                    properties={"label": str(index)},
                    vector={"bench": vector.tolist()},
                )
        failed = benchmark_collection.batch.failed_objects
        if failed:
            raise RuntimeError(f"{variant}: batch lỗi, ví dụ: {failed[0]}")
        build_seconds = time.perf_counter() - build_started

        latencies_ms = []
        recalls = []
        for query_index, query_vector in enumerate(BENCH_QUERY_VECTORS):
            query_started = time.perf_counter()
            result = benchmark_collection.query.near_vector(
                near_vector=query_vector.tolist(),
                target_vector="bench",
                limit=BENCH_TOP_K,
            )
            latencies_ms.append((time.perf_counter() - query_started) * 1000)
            actual_ids = {str(obj.uuid) for obj in result.objects}
            exact_ids = {BENCH_IDS[index] for index in exact_top_k[query_index]}
            recalls.append(len(actual_ids & exact_ids) / BENCH_TOP_K)

        sorted_latencies = sorted(latencies_ms)
        benchmark_results.append({
            "index": variant,
            "build_s": build_seconds,
            "recall@10": statistics.mean(recalls),
            "query_median_ms": statistics.median(sorted_latencies),
            "query_p95_ms": sorted_latencies[int(0.95 * (len(sorted_latencies) - 1))],
        })
finally:
    for collection_name in created_benchmark_collections:
        client.collections.delete(collection_name)

print(f"\nBộ dữ liệu: {BENCH_SIZE} vectors x {BENCH_DIMENSION} chiều; {BENCH_QUERY_COUNT} truy vấn.")
print(f"{'Index':<14} {'Build(s)':>10} {'Recall@10':>12} {'Median(ms)':>12} {'P95(ms)':>10}")
for row in benchmark_results:
    print(f"{row['index']:<14} {row['build_s']:>10.2f} {row['recall@10']:>12.3f} "
          f"{row['query_median_ms']:>12.2f} {row['query_p95_ms']:>10.2f}")


Bộ dữ liệu: 1500 vectors x 384 chiều; 24 truy vấn.
Index            Build(s)    Recall@10   Median(ms)    P95(ms)
HNSW M=8             0.63        0.679         3.63       4.79
HNSW M=16            0.57        0.871         4.03       5.00
HNSW M=32            0.69        0.967         4.98       6.16
Flat exact           0.69        1.000         7.04       8.01


In [18]:
collection = client.collections.get(COLLECTION_NAME)
config = collection.config.get()
vector_config = config.vector_config["default"]
index_config = vector_config.vector_index_config

print(f"Collection: {config.name}")
print("Properties:")
for prop in config.properties:
    print(f"  - {prop.name}: {prop.data_type.value}")
print(f"Vectorizer: {vector_config.vectorizer.vectorizer.value}")
print(f"Fields được vector hóa: {vector_config.vectorizer.source_properties}")
print(f"Index: HNSW | distance={index_config.distance_metric.value} | "
      f"max_connections={index_config.max_connections} | "
      f"ef_construction={index_config.ef_construction} | ef={index_config.ef}")
print("\nThông điệp chính: Weaviate quản lý schema, vector và vector index cùng collection.")

Collection: SeminarTechArticle
Properties:
  - title: text
  - content: text
  - category: text
  - published_year: int
Vectorizer: text2vec-transformers
Fields được vector hóa: ['title', 'content']
Index: HNSW | distance=cosine | max_connections=32 | ef_construction=128 | ef=64

Thông điệp chính: Weaviate quản lý schema, vector và vector index cùng collection.


C:\Users\hienm\AppData\Roaming\Python\Python313\site-packages\weaviate\warnings.py:355: ResourceWarning: Con004: The connection to Weaviate was not closed properly. This can lead to memory leaks.
            Please make sure to close the connection using `client.close()`.
  warnings.warn(
C:\Users\hienm\AppData\Local\Temp\ipykernel_12728\557112052.py:1: ResourceWarning: unclosed <socket.socket fd=1056, family=2, type=1, proto=0, laddr=('127.0.0.1', 65092), raddr=('127.0.0.1', 8080)>
  collection = client.collections.get(COLLECTION_NAME)


## 5. Nạp object theo batch

In [19]:
import time
import uuid

DEMO_DOCUMENTS = [
    {
        "title": "Vector Databases and Semantic Search",
        "content": "A vector database stores numerical embeddings for text, images, and other data. Similarity search retrieves items whose meaning is close to a natural-language query, even when they use different words.",
        "category": "Database",
        "published_year": 2025,
    },
    {
        "title": "HNSW for Approximate Nearest Neighbors",
        "content": "Hierarchical Navigable Small World graphs organize vectors into navigable layers. HNSW makes approximate nearest neighbor search fast for large embedding collections while balancing recall, memory, and latency.",
        "category": "Search",
        "published_year": 2024,
    },
    {
        "title": "Hybrid Search with BM25 and Dense Vectors",
        "content": "BM25 ranks documents using matching terms and their frequency. Dense vector search captures semantic similarity. Hybrid search combines keyword and vector signals; alpha controls the balance between them.",
        "category": "Search",
        "published_year": 2025,
    },
    {
        "title": "Retrieval-Augmented Generation (RAG)",
        "content": "A RAG system retrieves relevant source passages for a question, adds those passages to a prompt, and asks a language model to answer using the supplied context. Citations help readers verify the answer against its sources.",
        "category": "Generative AI",
        "published_year": 2025,
    },
    {
        "title": "Embeddings and Cosine Similarity",
        "content": "An embedding maps an item to a vector in a high-dimensional space. Cosine distance compares vector directions and is commonly used to rank semantically similar text.",
        "category": "Machine Learning",
        "published_year": 2023,
    },
    {
        "title": "Keeping RAG Knowledge Bases Fresh",
        "content": "Production retrieval systems need an update strategy. Stable document identifiers make it possible to detect existing records, refresh changed content, and avoid duplicate ingestion.",
        "category": "Data Engineering",
        "published_year": 2024,
    },
    {
        "title": "Access Control for Knowledge Retrieval",
        "content": "A retrieval system should enforce authorization before returning passages. Apply tenant and access-control filters during retrieval so private documents are not exposed in generated answers.",
        "category": "Security",
        "published_year": 2025,
    },
    {
        "title": "Computer Vision with Neural Networks",
        "content": "Convolutional neural networks learn visual features for image classification, object detection, and image segmentation tasks.",
        "category": "Machine Learning",
        "published_year": 2022,
    },
    {
        "title": "Building Modern React Interfaces",
        "content": "React applications compose reusable components and hooks to build responsive web interfaces with predictable state updates.",
        "category": "Frontend",
        "published_year": 2024,
    },
]

DEMO_NAMESPACE = uuid.UUID("9be3de4e-7c5e-46c7-bbe9-662676767e78")

def ingest_demo_documents(client: weaviate.WeaviateClient):
    collection = client.collections.get(COLLECTION_NAME)
    pending = []

    # Stable UUIDs make rerunning this demo safe: existing sample documents are skipped.
    for document in DEMO_DOCUMENTS:
        object_id = str(uuid.uuid5(DEMO_NAMESPACE, document["title"]))
        if collection.query.fetch_object_by_id(uuid=object_id) is None:
            pending.append((object_id, document))

    started_at = time.perf_counter()
    if pending:
        with collection.batch.dynamic() as batch:
            for object_id, document in pending:
                batch.add_object(uuid=object_id, properties=document)

        failed_objects = collection.batch.failed_objects
        if failed_objects:
            raise RuntimeError(
                f"Batch thất bại với {len(failed_objects)} object; "
                f"lỗi đầu tiên: {failed_objects[0]}"
            )

    elapsed = time.perf_counter() - started_at
    skipped = len(DEMO_DOCUMENTS) - len(pending)
    print(f"Đã thêm: {len(pending)} | Đã có sẵn: {skipped} | Thời gian: {elapsed:.2f}s")
    print(f"Tổng số tài liệu mẫu: {len(DEMO_DOCUMENTS)}")

ingest_demo_documents(client)

Đã thêm: 0 | Đã có sẵn: 9 | Thời gian: 0.00s
Tổng số tài liệu mẫu: 9


## 6. Xem vector, CRUD object và thống kê collection

In [20]:
collection = client.collections.get(COLLECTION_NAME)

# Đọc vector đã được Weaviate tạo tự động cho một document mẫu.
sample_id = str(uuid.uuid5(DEMO_NAMESPACE, DEMO_DOCUMENTS[0]["title"]))
sample = collection.query.fetch_object_by_id(uuid=sample_id, include_vector=True)
if sample is None:
    raise RuntimeError("Không tìm thấy object mẫu; hãy chạy lại cell nạp batch trước.")

vectors = sample.vector or {}
default_vector = vectors.get("default") if isinstance(vectors, dict) else vectors
if default_vector is None:
    raise RuntimeError("Không tìm thấy vector default của object.")
print(f"UUID: {sample.uuid}")
print(f"Object: {sample.properties['title']}")
print(f"Vector: {len(default_vector)} chiều; chỉ in vài giá trị đầu: {default_vector[:5]}")

# Aggregate cho thấy Weaviate có thể thống kê trực tiếp theo property.
summary = collection.aggregate.over_all(group_by="category", total_count=True)
print("\nSố object theo category:")
for group in sorted(summary.groups, key=lambda item: item.grouped_by.value):
    print(f"  {group.grouped_by.value}: {group.total_count}")

# CRUD trên một object tạm, không sửa các document mẫu.
CRUD_DEMO_UUID = str(uuid.uuid5(DEMO_NAMESPACE, "__crud_lifecycle_demo__"))
if collection.query.fetch_object_by_id(uuid=CRUD_DEMO_UUID) is not None:
    collection.data.delete_by_id(CRUD_DEMO_UUID)

try:
    created_uuid = collection.data.insert(
        uuid=CRUD_DEMO_UUID,
        properties={
            "title": "Temporary CRUD demonstration",
            "content": "This temporary object demonstrates Weaviate create, read, update, and delete operations.",
            "category": "Demo",
            "published_year": 2026,
        },
    )
    created = collection.query.fetch_object_by_id(uuid=created_uuid, include_vector=True)
    if created is None:
        raise RuntimeError("Không đọc lại được object vừa tạo.")
    print(f"\nCREATE + READ: {created.properties['title']} | UUID={created.uuid}")

    vector_before = created.vector["default"]
    collection.data.update(
        uuid=created_uuid,
        properties={
            "content": "The content changed; Weaviate re-vectorizes configured source properties on update."
        },
    )
    updated = collection.query.fetch_object_by_id(uuid=created_uuid, include_vector=True)
    if updated is None:
        raise RuntimeError("Không đọc lại được object sau khi cập nhật.")
    vector_after = updated.vector["default"]
    print(f"UPDATE: {updated.properties['content']}")
    print(f"Vector thay đổi sau khi sửa content: {vector_before != vector_after}")
finally:
    deleted = collection.data.delete_by_id(CRUD_DEMO_UUID)
    if not deleted:
        raise RuntimeError("Không xóa được object CRUD demo.")

if collection.query.fetch_object_by_id(uuid=CRUD_DEMO_UUID) is not None:
    raise RuntimeError("Object CRUD demo vẫn còn sau khi xóa.")
print("DELETE: đã xác nhận object CRUD demo không còn trong collection.")

UUID: 9f1854a2-dd21-516c-a664-7fdae562346a
Object: Vector Databases and Semantic Search
Vector: 384 chiều; chỉ in vài giá trị đầu: [0.10455117374658585, -0.12739016115665436, -0.08013882488012314, -0.06164516136050224, 0.00124513974878937]

Số object theo category:
  Data Engineering: 1
  Database: 1
  Frontend: 1
  Generative AI: 1
  Machine Learning: 2
  Search: 2
  Security: 1

CREATE + READ: Temporary CRUD demonstration | UUID=6ac3984c-e6e8-575d-a074-6ca887b37fcc
UPDATE: The content changed; Weaviate re-vectorizes configured source properties on update.
Vector thay đổi sau khi sửa content: True
DELETE: đã xác nhận object CRUD demo không còn trong collection.


## 7. So sánh semantic, keyword và hybrid search
Cùng một câu hỏi được gửi tới ba phương thức: near_text dùng embedding, bm25 dùng từ khóa, hybrid kết hợp hai tín hiệu. alpha điều chỉnh tỷ trọng giữa vector và keyword.

In [21]:
import weaviate.classes.query as wvq

def show_results(label: str, result, show_distance: bool = False):
    print(f"\n{label}")
    if not result.objects:
        print("  Không có kết quả")
        return

    for rank, obj in enumerate(result.objects, start=1):
        properties = obj.properties
        line = (
            f"  {rank}. {properties['title']} "
            f"[{properties['category']}, {properties['published_year']}]"
        )
        if show_distance and obj.metadata.distance is not None:
            line += f" | distance={obj.metadata.distance:.4f} (nhỏ hơn thường gần hơn)"
        print(line)

def compare_search_methods(client: weaviate.WeaviateClient):
    collection = client.collections.get(COLLECTION_NAME)
    query = "How does HNSW find similar items quickly in a large vector collection?"
    print(f"Câu hỏi: {query}")

    semantic = collection.query.near_text(
        query=query,
        limit=3,
        return_metadata=wvq.MetadataQuery(distance=True),
    )
    keyword = collection.query.bm25(query=query, limit=3)
    hybrid = collection.query.hybrid(query=query, alpha=0.5, limit=3)

    show_results("A. Semantic / vector search", semantic, show_distance=True)
    show_results("B. Keyword search / BM25", keyword)
    show_results("C. Hybrid search / alpha=0.5", hybrid)

    print("\nGợi ý trình bày: BM25 nhạy với từ xuất hiện; vector search tìm ý nghĩa gần; hybrid trộn cả hai.")

compare_search_methods(client)

Câu hỏi: How does HNSW find similar items quickly in a large vector collection?

A. Semantic / vector search
  1. HNSW for Approximate Nearest Neighbors [Search, 2024] | distance=0.2786 (nhỏ hơn thường gần hơn)
  2. Hybrid Search with BM25 and Dense Vectors [Search, 2025] | distance=0.4945 (nhỏ hơn thường gần hơn)
  3. Vector Databases and Semantic Search [Database, 2025] | distance=0.5035 (nhỏ hơn thường gần hơn)

B. Keyword search / BM25
  1. HNSW for Approximate Nearest Neighbors [Search, 2024]
  2. Embeddings and Cosine Similarity [Machine Learning, 2023]
  3. Vector Databases and Semantic Search [Database, 2025]

C. Hybrid search / alpha=0.5
  1. HNSW for Approximate Nearest Neighbors [Search, 2024]
  2. Embeddings and Cosine Similarity [Machine Learning, 2023]
  3. Vector Databases and Semantic Search [Database, 2025]

Gợi ý trình bày: BM25 nhạy với từ xuất hiện; vector search tìm ý nghĩa gần; hybrid trộn cả hai.


### Đánh giá hybrid search bằng tập truy vấn có đáp án mẫu
Một vài câu hỏi có một tài liệu liên quan được gán trước. Recall@3 đo tỷ lệ tìm thấy tài liệu đúng trong top 3; MRR@3 ưu tiên tài liệu đúng xuất hiện càng sớm càng tốt. Điểm số chỉ phản ánh tập demo nhỏ này.

In [22]:
EVALUATION_CASES = [
    {
        "query": "find fast approximate nearest neighbor lookup with layered graphs",
        "relevant": {"HNSW for Approximate Nearest Neighbors"},
    },
    {
        "query": "combine exact term matching with semantic meaning",
        "relevant": {"Hybrid Search with BM25 and Dense Vectors"},
    },
    {
        "query": "how retrieval augmented generation uses retrieved source passages",
        "relevant": {"Retrieval-Augmented Generation (RAG)"},
    },
    {
        "query": "angular similarity between normalized vector representations",
        "relevant": {"Embeddings and Cosine Similarity"},
    },
    {
        "query": "prevent private knowledge documents from being returned to unauthorized users",
        "relevant": {"Access Control for Knowledge Retrieval"},
    },
    {
        "query": "classify images and segment visual data with convolution",
        "relevant": {"Computer Vision with Neural Networks"},
    },
]

def evaluate_retriever(collection, method_name: str):
    recalls = []
    reciprocal_ranks = []
    for case in EVALUATION_CASES:
        if method_name == "semantic":
            response = collection.query.near_text(query=case["query"], limit=3)
        elif method_name == "bm25":
            response = collection.query.bm25(query=case["query"], limit=3)
        elif method_name == "hybrid":
            response = collection.query.hybrid(query=case["query"], alpha=0.5, limit=3)
        else:
            raise ValueError(f"Retriever không được hỗ trợ: {method_name}")

        titles = [obj.properties["title"] for obj in response.objects]
        relevant = case["relevant"]
        recalls.append(float(any(title in relevant for title in titles)))
        reciprocal_rank = next(
            (1 / rank for rank, title in enumerate(titles, start=1) if title in relevant),
            0.0,
        )
        reciprocal_ranks.append(reciprocal_rank)

    return statistics.mean(recalls), statistics.mean(reciprocal_ranks)

collection = client.collections.get(COLLECTION_NAME)
print(f"Đánh giá trên {len(EVALUATION_CASES)} truy vấn; relevant document được gán thủ công.")
print(f"{'Method':<12} {'Recall@3':>10} {'MRR@3':>10}")
for method in ("semantic", "bm25", "hybrid"):
    recall, mrr = evaluate_retriever(collection, method)
    print(f"{method:<12} {recall:>10.3f} {mrr:>10.3f}")

Đánh giá trên 6 truy vấn; relevant document được gán thủ công.
Method         Recall@3      MRR@3


semantic          1.000      1.000
bm25              1.000      0.722
hybrid            1.000      0.833


## 8. Retrieval cho RAG (không cần LLM)
Weaviate thực hiện phần retrieval: tìm bằng ngữ nghĩa, giới hạn theo metadata rồi tạo context có trích dẫn nguồn. Notebook dừng ở đây; không cần Ollama, model sinh câu trả lời hay API key.

In [23]:
def retrieve_rag_sources(client: weaviate.WeaviateClient, question: str, limit: int = 3):
    collection = client.collections.get(COLLECTION_NAME)
    result = collection.query.near_text(
        query=question,
        filters=wvq.Filter.by_property("published_year").greater_than(2023),
        limit=limit,
    )
    if not result.objects:
        raise RuntimeError("Không tìm thấy tài liệu để dựng context RAG.")

    source_blocks = []
    for index, obj in enumerate(result.objects, start=1):
        props = obj.properties
        source_blocks.append(
            f"[{index}] {props['title']} ({props['published_year']})\n{props['content']}"
        )
    return result.objects, "\n\n".join(source_blocks)

question = "How can a RAG application answer questions using trusted documents?"
rag_sources, rag_context = retrieve_rag_sources(client, question)
print("Nguồn được Weaviate truy xuất:")
print(rag_context)

Nguồn được Weaviate truy xuất:
[1] Access Control for Knowledge Retrieval (2025)
A retrieval system should enforce authorization before returning passages. Apply tenant and access-control filters during retrieval so private documents are not exposed in generated answers.

[2] Retrieval-Augmented Generation (RAG) (2025)
A RAG system retrieves relevant source passages for a question, adds those passages to a prompt, and asks a language model to answer using the supplied context. Citations help readers verify the answer against its sources.

[3] Keeping RAG Knowledge Bases Fresh (2024)
Production retrieval systems need an update strategy. Stable document identifiers make it possible to detect existing records, refresh changed content, and avoid duplicate ingestion.


## 9. Cùng một vector workload: Weaviate vs Milvus, Pinecone tùy chọn
Benchmark dùng cùng vector 384 chiều, cùng query và Recall@10 để so sánh công bằng hơn. Tập dữ liệu tổng hợp nhỏ chỉ so sánh đường truy vấn/index; không đại diện khác biệt về quản trị, tính năng, chi phí hoặc production scale.

Milvus chỉ chạy khi service profile comparison đang hoạt động. Pinecone tạo index serverless tạm và tự xóa khi hoàn tất; chỉ bật sau khi đã cấu hình key, cloud/region và chấp nhận quota/chi phí của tài khoản.

In [24]:
import socket

DB_BENCH_SIZE = 1000
DB_QUERY_COUNT = 20
DB_TOP_K = 10
DB_NAMESPACE = uuid.UUID("f5e40550-29d7-4240-b98c-cf0a3f1746e5")
db_rng = np.random.default_rng(2026)
DB_VECTORS = db_rng.standard_normal((DB_BENCH_SIZE, BENCH_DIMENSION), dtype=np.float32)
DB_VECTORS /= np.linalg.norm(DB_VECTORS, axis=1, keepdims=True)
DB_QUERY_VECTORS = db_rng.standard_normal(
    (DB_QUERY_COUNT, BENCH_DIMENSION), dtype=np.float32
)
DB_QUERY_VECTORS /= np.linalg.norm(DB_QUERY_VECTORS, axis=1, keepdims=True)
DB_EXACT = np.argsort(-(DB_QUERY_VECTORS @ DB_VECTORS.T), axis=1)[:, :DB_TOP_K]
DB_IDS = [str(index) for index in range(DB_BENCH_SIZE)]
DB_UUIDS = [str(uuid.uuid5(DB_NAMESPACE, item_id)) for item_id in DB_IDS]

def score_benchmark(name: str, search_function):
    latencies = []
    recalls = []
    for query_index, query_vector in enumerate(DB_QUERY_VECTORS):
        started = time.perf_counter()
        result_ids = search_function(query_vector)
        latencies.append((time.perf_counter() - started) * 1000)
        expected = {DB_IDS[index] for index in DB_EXACT[query_index]}
        recalls.append(len(set(result_ids) & expected) / DB_TOP_K)
    ordered = sorted(latencies)
    return {
        "database": name,
        "recall@10": statistics.mean(recalls),
        "median_ms": statistics.median(ordered),
        "p95_ms": ordered[int(0.95 * (len(ordered) - 1))],
    }

def port_is_open(host: str, port: int) -> bool:
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False

comparison_results = []
created_db_collection = f"SeminarDbBench{uuid.uuid4().hex[:10]}"
db_collection = None
try:
    db_collection = client.collections.create(
        name=created_db_collection,
        vector_config=wvc.Configure.Vectors.self_provided(
            name="bench",
            vector_index_config=wvc.Configure.VectorIndex.hnsw(
                distance_metric=wvc.VectorDistances.COSINE,
                max_connections=16,
                ef_construction=128,
                ef=64,
            ),
        ),
        properties=[wvc.Property(name="label", data_type=wvc.DataType.TEXT)],
    )
    with db_collection.batch.dynamic() as batch:
        for index, vector in enumerate(DB_VECTORS):
            batch.add_object(
                uuid=DB_UUIDS[index],
                properties={"label": DB_IDS[index]},
                vector={"bench": vector.tolist()},
            )
    failures = db_collection.batch.failed_objects
    if failures:
        raise RuntimeError(f"Weaviate benchmark batch thất bại: {failures[0]}")

    comparison_results.append(score_benchmark(
        "Weaviate",
        lambda query: [
            obj.properties["label"]
            for obj in db_collection.query.near_vector(
                near_vector=query.tolist(), target_vector="bench", limit=DB_TOP_K
            ).objects
        ],
    ))
finally:
    if client.collections.exists(created_db_collection):
        client.collections.delete(created_db_collection)

if port_is_open("127.0.0.1", 19530):
    from pymilvus import DataType, MilvusClient

    milvus = MilvusClient(uri="http://127.0.0.1:19530")
    milvus_collection_name = f"seminar_bench_{uuid.uuid4().hex[:10]}"
    try:
        schema = MilvusClient.create_schema(auto_id=False, enable_dynamic_field=False)
        schema.add_field(field_name="id", datatype=DataType.VARCHAR, max_length=64, is_primary=True)
        schema.add_field(field_name="vector", datatype=DataType.FLOAT_VECTOR, dim=BENCH_DIMENSION)
        index_params = milvus.prepare_index_params()
        index_params.add_index(
            field_name="vector",
            index_type="HNSW",
            metric_type="COSINE",
            params={"M": 16, "efConstruction": 128},
        )
        milvus.create_collection(
            collection_name=milvus_collection_name,
            schema=schema,
            index_params=index_params,
        )
        milvus.insert(
            collection_name=milvus_collection_name,
            data=[{"id": DB_IDS[i], "vector": DB_VECTORS[i].tolist()} for i in range(DB_BENCH_SIZE)],
        )
        milvus.load_collection(collection_name=milvus_collection_name)

        def milvus_search(query):
            hits = milvus.search(
                collection_name=milvus_collection_name,
                data=[query.tolist()],
                anns_field="vector",
                search_params={"metric_type": "COSINE", "params": {"ef": 64}},
                limit=DB_TOP_K,
                output_fields=["id"],
            )[0]
            return [hit["id"] for hit in hits]

        comparison_results.append(score_benchmark("Milvus", milvus_search))
    finally:
        if milvus.has_collection(milvus_collection_name):
            milvus.drop_collection(collection_name=milvus_collection_name)
        milvus.close()
else:
    print("Milvus bỏ qua: chưa chạy. Bật bằng docker compose --profile comparison up -d.")

if RUN_PINECONE_COMPARISON:
    pinecone_api_key = os.getenv("PINECONE_API_KEY")
    if not pinecone_api_key:
        raise RuntimeError("RUN_PINECONE_COMPARISON=1 nhưng thiếu PINECONE_API_KEY.")

    from pinecone import Pinecone, ServerlessSpec

    pinecone_client = Pinecone(api_key=pinecone_api_key)
    pinecone_index_name = f"seminar-{uuid.uuid4().hex[:8]}"
    pinecone_created = False
    try:
        pinecone_client.create_index(
            name=pinecone_index_name,
            dimension=BENCH_DIMENSION,
            metric="cosine",
            spec=ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION),
        )
        pinecone_created = True
        while not pinecone_client.describe_index(pinecone_index_name).status.ready:
            time.sleep(2)
        pinecone_index = pinecone_client.Index(pinecone_index_name)

        for start in range(0, DB_BENCH_SIZE, 100):
            end = min(start + 100, DB_BENCH_SIZE)
            pinecone_index.upsert(vectors=[
                {"id": DB_IDS[i], "values": DB_VECTORS[i].tolist()}
                for i in range(start, end)
            ])

        def pinecone_search(query):
            response = pinecone_index.query(vector=query.tolist(), top_k=DB_TOP_K)
            return [match.id for match in response.matches]

        comparison_results.append(score_benchmark("Pinecone", pinecone_search))
    finally:
        if pinecone_created:
            pinecone_client.delete_index(pinecone_index_name)
else:
    print("Pinecone bỏ qua: cloud benchmark đang tắt (RUN_PINECONE_COMPARISON=0).")

print(f"\nCùng workload: {DB_BENCH_SIZE} vectors x {BENCH_DIMENSION} chiều; {DB_QUERY_COUNT} truy vấn.")
print(f"{'Database':<12} {'Recall@10':>12} {'Median(ms)':>12} {'P95(ms)':>10}")
for row in comparison_results:
    print(f"{row['database']:<12} {row['recall@10']:>12.3f} "
          f"{row['median_ms']:>12.2f} {row['p95_ms']:>10.2f}")

Milvus bỏ qua: chưa chạy. Bật bằng docker compose --profile comparison up -d.
Pinecone bỏ qua: cloud benchmark đang tắt (RUN_PINECONE_COMPARISON=0).

Cùng workload: 1000 vectors x 384 chiều; 20 truy vấn.
Database        Recall@10   Median(ms)    P95(ms)
Weaviate            0.955         4.63       5.53


## 10. Tổng kết
Weaviate quản lý collection, object, vectorizer và HNSW; hỗ trợ batch, CRUD, aggregate, vector/BM25/hybrid query và filter. Các benchmark đã cho thấy cách đo recall/latency trên cùng bộ dữ liệu, không phải kết luận tổng quát cho mọi môi trường. RAG có thể dùng Weaviate để truy xuất và dựng context; LLM chỉ cần nếu muốn sinh câu trả lời.

Không chia sẻ Docker port anonymous-access này ra Internet.

In [25]:
client.close()
print("Đã đóng kết nối Weaviate.")

Đã đóng kết nối Weaviate.
